# CD CosMx Resection: Raw Data Processing (12 samples)

QC, normalization, and tissue section DBSCAN assignment for all 12 resection samples.
AUCell annotation: `04_resection_aucell_annotation.R`.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import os
import gc
from scipy.spatial.distance import pdist
from sklearn.cluster import DBSCAN

# reading in data and normalization

In [ ]:
samples = [
'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:
prefix = '/path/to/cosmx_data/cd_resection/'

In [ ]:
# prefix = directory with one folder per run (AtoMx flat files in <run>/dir_formatted/); outputs go to <run>/Processed/

for i in samples:
    sample_dir_write = prefix+i+'/Processed/'
    sample_dir = prefix+i+'/dir_formatted'
    fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
    fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
    if 'FOV' in fov_df.columns:
      # Rename 'FOV' column to 'fov'
      fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
      # have fov_file reference the new, formatted file and write it
      fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
      fov_df.to_csv(fov_file, index=False)
    
    for i in os.listdir(sample_dir):
        if 'exprMat' in i:
            sample = i.split('_exprMat_file.csv.gz')[0]

    adata = sq.read.nanostring(
        path=sample_dir,
        counts_file=sample+"_exprMat_file.csv.gz",
        meta_file=sample+"_metadata_file.csv.gz",
        fov_file="fov_positions_formatted.csv")

    to_drop = []
    for i in adata.obs.columns:
        if 'clust' in i:
            to_drop.append(i)

    adata.obs = adata.obs.drop(to_drop,axis = 1)
    os.makedirs(sample_dir_write, exist_ok=True)
    adata.write_h5ad(sample_dir_write+'raw_data.h5ad')


In [ ]:
# raw counts, sample name, negative and system-control probes removed, QC metrics
for sample in samples:
    sample_dir_write = prefix+sample+'/Processed/'
    adata = sc.read_h5ad(sample_dir_write+'raw_data.h5ad')

    # add in metadata
    adata.obs['sample'] = sample

    # remove negative probes and control
    adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
    adata.var['control_probe']=[True if 'SystemControl' in i else False for i in adata.var.index.tolist()]
    adata=adata[:, ~adata.var["neg_probe"]]
    adata=adata[:, ~adata.var["control_probe"]]

    # qc
    sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)

In [ ]:
# qc thresholds for filtering
ngenes_top= 2500
ngenes_bot= 50
ncounts_top= 5000
ncounts_bot= 100
area_top = 30000

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=ngenes_bot)
sc.pp.filter_cells(adata, min_genes=ngenes_bot)
sc.pp.filter_cells(adata, max_counts=ncounts_top)
sc.pp.filter_cells(adata, max_genes=ngenes_top)
adata=adata[adata.obs['Area']<=area_top]

# norm+log1p 
adata.layers['raw']=adata.X.toarray()
sc.pp.normalize_total(adata, inplace=True)
count = pd.DataFrame(adata.X.toarray())
count.columns = adata.var.index.tolist()
count.index=adata.obs.index.tolist()
count.to_csv(sample_dir_write+'norm_counts.csv')
sc.pp.log1p(adata)
gc.collect()

In [ ]:
def assign_tissue_sections(adata, fov_column='fov', spatial_key='spatial_fov', eps=None, min_samples=1, distance_percentile=5):
    """Tissue section per FOV: DBSCAN on FOV centroids; eps = percentile of the pairwise centroid distances if not given."""
    coords = adata.obsm[spatial_key]
    fovs = adata.obs[fov_column].unique()

    # FOV centroids
    fov_centroids = []
    fov_labels = []
    for fov in fovs:
        fov_mask = adata.obs[fov_column] == fov
        fov_centroids.append(coords[fov_mask].mean(axis=0))
        fov_labels.append(fov)
    fov_centroids = np.array(fov_centroids)

    if eps is None:
        eps = np.percentile(pdist(fov_centroids), distance_percentile)
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit(fov_centroids).labels_
    fov_to_tissue = dict(zip(fov_labels, labels))

    # noise points (label -1) become separate tissues
    if -1 in labels:
        noise_fovs = [fov for fov, label in fov_to_tissue.items() if label == -1]
        max_label = max([l for l in labels if l != -1]) if any(l != -1 for l in labels) else -1
        for i, fov in enumerate(noise_fovs):
            fov_to_tissue[fov] = max_label + 1 + i

    adata.obs['tissue_section'] = adata.obs[fov_column].map(fov_to_tissue)
    adata.obs['tissue_section'] = 'Tissue_' + adata.obs['tissue_section'].astype(str)
    return adata

In [ ]:
adata = assign_tissue_sections(adata, fov_column='fov', spatial_key='spatial_fov',distance_percentile=3)

In [ ]:
adata.write_h5ad(sample_dir_write+'norm_log1p.h5ad')